# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All record sets, fields, and columns are referenced by their `@id`. This ensures reproducibility and consistent referencing across analyses.

In [ ]:
# List all available record sets and their details

from collections.abc import Iterable

def list_recordsets_fields(ds):
    print("--- Available Record Sets ---")
    for rs in ds.record_sets:
        print(f"@id: {rs['@id']}")
        print(f"  name: {rs.get('name', '')}")
        print(f"  description: {rs.get('description', '')}")
        # List fields for this record set
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], Iterable) and not isinstance(rs['field'], str) else [rs['field']]
            print("  Fields:")
            for f in fields:
                print(f"    - @id: {f['@id']}, name: {f.get('name','')}, dataType: {f.get('dataType','')}")
        print()
list_recordsets_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we'll extract the data from all available record sets. Replace the `record_sets` list with specific `@id`s as needed.

In [ ]:
# Extract data from each available record set into DataFrames
import pprint
from collections.abc import Iterable

# Get all available record set @ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from Record Set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"  No records found for {record_set_id}.")
        else:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
            display(dataframes[record_set_id].head())
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")
        continue

# If at least one record set loaded, print its fields and head
if dataframes:
    main_record_set_id = list(dataframes)[0]
    print(f"Using Record Set '@id': {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, or grouping data by key attributes to prepare for analysis.

**Note**: All field selections reference their `@id` for clarity and reproducibility.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Pick a representative record set and candidate numeric field
if not dataframes:
    raise ValueError("No dataframes loaded. Please check previous steps.")

# You may wish to update these IDs from the output of the overview step above for your dataset
record_set_id = main_record_set_id
df = dataframes[record_set_id]
pprint.pprint(df.columns.tolist())

# Try infering a numeric field (@id) for demo purposes -- choose the first float or integer column
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if not numeric_field:
    # Try to coerce columns that might be numbers but are object dtype
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue
if not numeric_field:
    print("No numeric field found in the first record set. Please adjust field selection in the EDA.")
else:
    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Modify the field selection as relevant to find interesting patterns!**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution, if available
if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If group_field exists, boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} distribution by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the dataset with Croissant schema using `mlcroissant`, examined available record sets and fields (by `@id`), and explored one of the data tables with basic analysis and visualizations.

You can continue by selecting further record sets, engineering features, or building models on top of these curated, well-annotated data tables.

**Note:** For robust results, always consult the Croissant metadata and data dictionary to interpret field meanings and units. All operations above reference data entities by their `@id` as recommended.